# Gradient Matching

实际上我们经常做这样一件事，就是指出优化的最小单位。比如 Flow Matching 指出优化模型输出的边缘矢量场最小单位是优化模型输出的条件矢量场，前者的 Ground-Truth 是几乎无法估计的，后者却可以被简单地写出。

Gradient Matching 的思想非常类似。我们发现原始数据集蒸馏的优化太过复杂，那么能否指出一个更小的优化单位？实际上这个最小单位完全存在，那就是梯度。

推荐你读 https://arxiv.org/abs/2006.05929 Dataset Condensation with Gradient Matching 这是 Gradient Matching 原文，同样是一篇开山之作。

我们详细说说。

# 基本逻辑

原始数据集蒸馏目标可以视为
$$\min_{\tilde{x}}
\ell\bigl(x,\theta_K(\tilde{x})\bigr)$$
我们希望蒸馏数据集可以促使模型优化时产生正确的梯度，并且根据该梯度优化之后的结果可以在真实数据集上表现良好。

那么为什么不直接对齐梯度本身？

我们换一个优化对象
$$\min_{\tilde{x}}
D\left(
\nabla_\theta\ell(x,\theta),
\nabla_\theta\ell(\tilde{x},\theta)
\right)$$
其中 $D(\cdot,\cdot)$ 是一个度量。

这里的思想非常简单。如果真实数据和合成数据在许多模型状态上都给出相近的下降方向，那么在合成数据上训练，应该近似在真实数据上训练。

我们开始正式地叙述。首先给定数据集 $\mathcal{T} = \{(x_i, y_i)\}$ 和网络 $\phi_\theta$，我们可以写出最优参数
$$\theta^\mathcal{T} = \arg\min_\theta \mathcal{L}^\mathcal{T}(\theta) = \frac{1}{|\mathcal{T}|} \sum_{(x,y)\in\mathcal{T}} \ell(\phi_\theta(x), y)$$
现在我们有蒸馏数据集 $\mathcal{S} = \{(s_i, y_i)\}, |\mathcal{S}| \ll |\mathcal{T}|$，我们写出最优参数
$$\theta^\mathcal{S} = \arg\min_\theta \mathcal{L}^\mathcal{S}(\theta)$$
我们希望最优参数 $\theta^\mathcal{S}$ 在真实数据集上表现好，这意味着我们要求蒸馏数据集是
$$\mathcal{S}^* = \arg\min_\mathcal{S} \mathcal{L}^\mathcal{T}(\theta^\mathcal{S}(\mathcal{S}))$$
以上是我们上一章讲述的朴素数据集蒸馏的思想。这里涉及到一个双层优化问题，导致计算非常昂贵。

现在我们逐步开始提出进阶的想法。首先，我们提出 Parameter Matching，我们希望蒸馏数据集优化参数和原始数据集优化参数可以直接匹配，这意味着他们的输出是相似的。换言之，我们希望
$$\min_{\mathcal{S}}
\mathbb{E}_{\theta_0\sim P_{\theta_0}}
\left[
D\left(
\theta^{\mathcal{S}}(\theta_0),
\theta^{\mathcal{T}}(\theta_0)
\right)
\right]$$
这还是一个昂贵的双层优化问题。因此我们提出，$\theta^{\mathcal{S}}$ 可以不是最优参数，而是经过固定步数优化后的最优
$$\theta^{\mathcal{S}}(\mathcal{S})
=
\operatorname{opt\text{-}alg}_{\theta}
\left(
\mathcal{L}^{\mathcal{S}}(\theta),
\varsigma
\right)$$
其中 $\operatorname{opt\text{-}alg}$ 表示某个优化算法，$\varsigma$ 表示固定优化步数。

Parameter Matching 的思想是，可以先训练好 $\theta^{\mathcal{T}}(\theta_0)$，然后从初始蒸馏数据集出发优化固定步数得到 $\theta^{\mathcal{S}}(\theta_0)$，然后计算我们需要的损失并且优化。

然而，一个巨大的问题是，训练初期 $\theta^{\mathcal{S}}$ 可能还距离 $\theta^{\mathcal{T}}$ 非常遥远，这意味对于损失 $D\left(
\theta^{\mathcal{S}},
\theta^{\mathcal{T}}
\right)$ 的优化过程充斥着噪声与局部极小值，目标跨度太大。

另一个巨大的问题是，即使两个神经网络实现完全相同的函数，它们的参数也可能差异很大。例如隐藏层神经元可以置换，若交换两个隐藏神经元，同时相应交换下一层权重，网络函数不变，但参数向量不同，这意味这个优化目标没有那么合理。

我们希望否决 Parameter Matching 并且提出其进阶。我们原本希望
$$\theta^{\mathcal{S}}
\approx
\theta^{\mathcal{T}}$$
现在改为
$$\nabla_\theta
\mathcal{L}^{\mathcal{S}}(\theta)
\approx
\nabla_\theta
\mathcal{L}^{\mathcal{T}}(\theta)$$
这就是 Gradient Matching 的基本思想。

下面这张图简单展示了这里的思想，无论是 Parameter Matching 还是 Gradient Matching，实际上都遵从右边这一套流程。

<img src="./assets/GM.png" width="700" height="250">

但是我们先不说 GM 的完整算法。一个重要技巧是 Curriculum Gradient Matching，简而言之就是我们希望解决上述优化梯度过大问题。我们提出，不只让合成数据训练出的模型最终接近 $\theta^\mathcal{T}$，还要沿着训练轨迹每一步都尽量接近对应参数轨迹 $\theta^\mathcal{T}_t$。

当我们对 PM 应用这一思想，优化目标变为
$$\min_{\mathcal{S}} \mathbb{E}_{\theta_0\sim P_{\theta_0}} \Big[ \sum_{t=0}^{T-1} D(\theta_t^\mathcal{S}, \theta_t^\mathcal{T}) \Big]$$
其中 $T$ 是迭代总步数，$\theta_t^\mathcal{S}$ 是在合成数据上训练 $t$ 步得到的参数，$\theta_t^\mathcal{T}$ 是是在真实数据上训练 $t$ 步得到的参数。

当我们对 GM 应用这一思想，首先我们写出梯度下降更新过程
$$\theta_{t+1}^\mathcal{S} \gets \theta_t^\mathcal{S} - \eta_\theta \nabla_\theta \mathcal{L}^\mathcal{S}(\theta_t^\mathcal{S})$$
$$\theta_{t+1}^\mathcal{T} \gets \theta_t^\mathcal{T} - \eta_\theta \nabla_\theta \mathcal{L}^\mathcal{T}(\theta_t^\mathcal{T})$$
最终优化目标是
$$\min_\mathcal{S} \mathbb{E}_{\theta_0 \sim P_{\theta_0}} \Big[ \sum_{t=0}^{T-1} D(\nabla_\theta \mathcal{L}^\mathcal{S}(\theta_t), \nabla_\theta \mathcal{L}^\mathcal{T}(\theta_t)) \Big]$$

这就是大概的思想。接下来我们说说具体算法。

# 蒸馏算法

给定权重初始化 $\theta_0 \sim P_{\theta_0}$。

对于迭代步数 $t=0,...,T-1$，对于每个类别 $c$ 抽取真实数据集 mini-batch $B_c^\mathcal{T}$ 和蒸馏数据集 mini-batch $B_c^\mathcal{S}$。我们可以计算梯度
$$\nabla_\theta \mathcal{L}_c^\mathcal{T}(\theta_t),\quad
\nabla_\theta \mathcal{L}_c^\mathcal{S}(\theta_t)$$
现在更新蒸馏数据集
$$\mathcal{S}_c \gets \text{opt-alg}_\mathcal{S}\Big(D(\nabla_\theta \mathcal{L}_c^\mathcal{S}, \nabla_\theta \mathcal{L}_c^\mathcal{T}),\varsigma_\mathcal{S},\eta_\mathcal{S}\Big)$$
然后更新 $\theta_{t+1}$
$$\theta_{t+1}
\leftarrow
\operatorname{opt\text{-}alg}_{\theta}
\left(
\mathcal{L}^{\mathcal{S}}(\theta_t),\varsigma_\theta,\eta_\theta
\right)$$
进入下一轮迭代。

关于这里的优化算法 $\text{opt-alg}$，实际上有很多选择，比如 SGD 就是
$$
\mathcal{S}_c
\leftarrow
\mathcal{S}_c
-
\eta_{\mathcal{S}}
\nabla_{\mathcal{S}_c}
D\left(
\nabla_\theta \mathcal{L}_c^{\mathcal{S}}(\theta_t),
\nabla_\theta \mathcal{L}_c^{\mathcal{T}}(\theta_t)
\right)
$$
$$
\theta_{t+1}
=
\theta_t
-
\eta_\theta
\nabla_{\theta_t}
\mathcal{L}^{\mathcal{S}}(\theta_t)$$
更多的，请注意一件事，更新到 $\theta_{t+1}$ 需要重新计算梯度 $\nabla_{\theta_t} \mathcal{L}^{\mathcal{S}}(\theta_t)$ 而不是直接复用上面已经得到的旧梯度。

现在我们详细说说这个度量 $D(\cdot,\cdot)$。神经网络中具有很多各种形状的层，我们需要定义一个距离衡量层之间差异。

其实非常简单，对于某层梯度，我们直接展平为一维。我们记我们需要比较的向量为 $A, B$，那么度量就是
$$D(A,B) = \sum_i \Big(1 - \frac{A_i \cdot B_i}{\|A_i\|\|B_i\|}\Big)$$

更详细的，对于全连接层，权重是 $W\in\mathbb{R}^{n_{\mathrm{out}}\times n_{\mathrm{in}}}$，梯度会保持形状 $\nabla_W\ell\in\mathbb{R}^{n_{\mathrm{out}}\times n_{\mathrm{in}}}$。
我们记第 $i$ 行是
$$\left(\nabla_W\ell\right)_{i,:}
\in
\mathbb{R}^{n_{\mathrm{in}}}$$
更多的，对于真实数据第 $i$ 行产生梯度记为 $B_i$，对于蒸馏数据集产生梯度记为 $A_i$。有
$$A_i,B_i\in\mathbb{R}^{n_{\mathrm{in}}}$$
所以最后度量就是
$$D(A,B)
=
\sum_{i=1}^{n_{\mathrm{out}}}
\left(
1-
\frac{A_i\cdot B_i}
{\|A_i\|\|B_i\|}
\right)$$

解释一下为什么要分行计算，原因是第 $i$ 个输出 $z_i$ 可以写成
$$z_i
=
\sum_{j=1}^{n_{\mathrm{in}}}
W_{ij}x_j+b_i$$
因此梯度 $\nabla_{W_{i,:}}\ell$ 对这一行损失负责。

所以在卷积神经网络和自注意力模块上也做相同的事情，就是关于输出维度展开，换言之展开的元素维度就是原维度除以输出维度。最后度量求和都是按照输出维度求和。

下面是完整的 GM 算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Dataset condensation with gradient matching} \\
\hline
\textbf{Input:} \text{ Training set } \mathcal{T} \\
\begin{aligned}
1: & \ \textbf{Required:} \text{ Randomly initialized set of synthetic samples } \mathcal{S} \text{ for } C \text{ classes, probability distribution over} \\
   & \ \text{randomly initialized weights } P_{\boldsymbol{\theta}_0}, \text{ deep neural network } \phi_{\boldsymbol{\theta}}, \text{ number of outer-loop steps } K, \text{ number of} \\
   & \ \text{inner-loop steps } T, \text{ number of steps for updating weights } \varsigma_{\boldsymbol{\theta}} \text{ and synthetic samples } \varsigma_{\mathcal{S}} \text{ in each inner-loop} \\
   & \ \text{step respectively, learning rates for updating weights } \eta_{\boldsymbol{\theta}} \text{ and synthetic samples } \eta_{\mathcal{S}}. \\
2: & \ \textbf{for } k = 0, \dots, K - 1 \textbf{ do} \\
3: & \ \quad \text{Initialize } \boldsymbol{\theta}_0 \sim P_{\boldsymbol{\theta}_0} \\
4: & \ \quad \textbf{for } t = 0, \dots, T - 1 \textbf{ do} \\
5: & \ \quad\quad \textbf{for } c = 0, \dots, C - 1 \textbf{ do} \\
6: & \ \quad\quad\quad \text{Sample a minibatch pair } B_c^{\mathcal{T}} \sim \mathcal{T} \text{ and } B_c^{\mathcal{S}} \sim \mathcal{S} \qquad \qquad \ \ \triangleright B_c^{\mathcal{T}} \text{ and } B_c^{\mathcal{S}} \text{ are of the same class } c. \\
7: & \ \quad\quad\quad \text{Compute } \mathcal{L}_c^{\mathcal{T}} = \frac{1}{|B_c^{\mathcal{T}}|} \sum_{(\boldsymbol{x}, y) \in B_c^{\mathcal{T}}} \ell(\phi_{\boldsymbol{\theta}_t}(\boldsymbol{x}), y) \text{ and } \mathcal{L}_c^{\mathcal{S}} = \frac{1}{|B_c^{\mathcal{S}}|} \sum_{(\boldsymbol{s}, y) \in B_c^{\mathcal{S}}} \ell(\phi_{\boldsymbol{\theta}_t}(\boldsymbol{s}), y) \\
8: & \ \quad\quad\quad \text{Update } \mathcal{S}_c \leftarrow \text{opt-alg}_{\mathcal{S}}\left(D(\nabla_{\boldsymbol{\theta}}\mathcal{L}_c^{\mathcal{S}}(\boldsymbol{\theta}_t), \nabla_{\boldsymbol{\theta}}\mathcal{L}_c^{\mathcal{T}}(\boldsymbol{\theta}_t)), \varsigma_{\mathcal{S}}, \eta_{\mathcal{S}}\right) \\
9: & \ \quad \quad \text{Update } \boldsymbol{\theta}_{t+1} \leftarrow \text{opt-alg}_{\boldsymbol{\theta}}(\mathcal{L}^{\mathcal{S}}(\boldsymbol{\theta}_t), \varsigma_{\boldsymbol{\theta}}, \eta_{\boldsymbol{\theta}}) \qquad \qquad \qquad \qquad \qquad \qquad \triangleright \text{Use the whole } \mathcal{S} \\
10:& \ \textbf{end for}
\end{aligned} \\
\textbf{Output:} \ \mathcal{S} \\
\hline
\end{array}$$